# 01 — 稀疏训练与 BN 通道剪枝

对应论文 **Figure 2** 与 Table 1：*YOLO V8 Network Lightweight Method Based on Pruning and Quantization* (Cheng, 2024)。

**流程：** 稀疏训练（BN L1）→ 结构化通道剪枝 → 重训练 → Params / GFLOPs / mAP 对比。

Paper Figure 2 / Table 1. Pipeline: BN L1 sparsity training → structured channel pruning → retrain → compare Params / GFLOPs / mAP.



## 0. 配置

`QUICK_DEMO=True` 用很少的 epoch 跑通流程。论文设置请改为 `False`（稀疏训练 50 epoch，重训练 100 epoch，数据换 `coco.yaml` 更接近论文表格）。



In [ ]:
import shutil
import sys
from pathlib import Path

import torch
from ultralytics import YOLO

ROOT = Path.cwd().resolve()
if not (ROOT / "src").is_dir():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from src.sparsity import BNSparsity
from src.prune import prune_yolov8
from src.eval_utils import model_stats, val_map

QUICK_DEMO = True          # False = paper-scale schedule
DATA = "coco128.yaml"      # paper used COCO: "coco.yaml"
BASE_WEIGHTS = "yolov8n.pt"
IMGSZ = 640
DEVICE = 0 if torch.cuda.is_available() else "cpu"

SPARSE_EPOCHS = 3 if QUICK_DEMO else 50
RETRAIN_EPOCHS = 3 if QUICK_DEMO else 100
KEEP_RATIO = 0.8
MIN_CHANNELS = 8

WEIGHTS = ROOT / "weights"
WEIGHTS.mkdir(exist_ok=True)
SPARSE_PT = WEIGHTS / "sparse.pt"
PRUNE_PT = WEIGHTS / "prune.pt"
RETRAIN_PT = WEIGHTS / "retrain.pt"

print("root:", ROOT)
print("device:", DEVICE, "| demo:", QUICK_DEMO)



## 1. 稀疏训练

论文对 BN 的 \(\gamma, \beta\) 加 L1（公式 5–9），使部分通道的 \(\gamma \to 0\)，再按阈值剪枝。

\(\lambda_1 = 0.01 \times (1 - 0.9 \cdot e / n_e)\)

原 ultralytics `trainer.py` 里这段是注释掉的；这里用 callback 补丁在 AMP `unscale` 之后写入梯度，不改库源码。



In [ ]:
yolo = YOLO(BASE_WEIGHTS)
sparsity = BNSparsity(epochs=SPARSE_EPOCHS, lambda1_0=0.01, lambda2=0.01, decay=0.9)
sparsity.register_yolo_callbacks(yolo)

yolo.train(
    data=DATA,
    epochs=SPARSE_EPOCHS,
    imgsz=IMGSZ,
    device=DEVICE,
    project=str(ROOT / "runs"),
    name="sparse",
    exist_ok=True,
    pretrained=True,
)

ckpt = ROOT / "runs" / "sparse" / "weights" / "last.pt"
if not ckpt.exists():
    ckpt = ROOT / "runs" / "sparse" / "weights" / "best.pt"
shutil.copy2(ckpt, SPARSE_PT)
print("sparse checkpoint:", SPARSE_PT)



## 2. 通道剪枝

全局收集 BN \(|\gamma|\)，按 `KEEP_RATIO` 取阈值。只剪论文列出的衔接处，跳过 stem、Detect 拼接层、C2f split 与 FPN upsample。



In [ ]:
sparse_src = SPARSE_PT if SPARSE_PT.exists() else BASE_WEIGHTS
prune_yolov8(sparse_src, PRUNE_PT, keep_ratio=KEEP_RATIO, min_channels=MIN_CHANNELS)
YOLO(str(PRUNE_PT)).info()



## 3. 剪枝后重训练

剪枝会掉点，论文用重训练把精度从 0.722 拉回到 0.754（Table 1，COCO）。



In [ ]:
yolo_p = YOLO(str(PRUNE_PT))
yolo_p.train(
    data=DATA,
    epochs=RETRAIN_EPOCHS,
    imgsz=IMGSZ,
    lr0=0.01,
    device=DEVICE,
    project=str(ROOT / "runs"),
    name="retrain",
    exist_ok=True,
)

ckpt = ROOT / "runs" / "retrain" / "weights" / "best.pt"
if not ckpt.exists():
    ckpt = ROOT / "runs" / "retrain" / "weights" / "last.pt"
shutil.copy2(ckpt, RETRAIN_PT)
print("retrain checkpoint:", RETRAIN_PT)



## 4. Table 1 对比

论文 COCO 结果（对照用，不是本 notebook 的输出）：

| | GFLOPs | Params | Acc |
|---|---:|---:|---:|
| Before | 8.7 | 3,151,904 | 0.771 |
| Prune | 5.8 | 2,536,321 | 0.722 |
| Retrain | — | — | 0.754 |



In [ ]:
def row(tag, pt):
    m = YOLO(str(pt))
    stats = model_stats(m)
    metrics = val_map(m, DATA)
    print(f"{tag:10} params={stats['params']:<12} gflops={stats['gflops']:<8} "
          f"mAP50={metrics['map50']:.4f}  mAP={metrics['map']:.4f}")
    return stats, metrics

print(f"{'stage':10} {'params':12} {'gflops':8}  mAP50 / mAP")
row("before", BASE_WEIGHTS)
if PRUNE_PT.exists():
    row("prune", PRUNE_PT)
if RETRAIN_PT.exists():
    row("retrain", RETRAIN_PT)

